# Run Any Kind of OLS Regression (ANOVA, GLM, etc.)

### Authors: Calvin Howard.

#### Last updated: July 6, 2023

Use this to run/test a statistical model (e.g., regression or T-tests) on a spreadsheet.

Notes:
- To best use this notebook, you should be familar with GLM design and Contrast Matrix design. See this webpage to get started:
[FSL's GLM page](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/GLM)

# 00 - Import CSV with All Data
**The CSV is expected to be in this format**
- ID and absolute paths to niftis are critical
```
+-----+----------------------------+--------------+--------------+--------------+
| ID  | Nifti_File_Path            | Covariate_1  | Covariate_2  | Covariate_3  |
+-----+----------------------------+--------------+--------------+--------------+
| 1   | /path/to/file1.nii.gz      | 0.5          | 1.2          | 3.4          |
| 2   | /path/to/file2.nii.gz      | 0.7          | 1.4          | 3.1          |
| 3   | /path/to/file3.nii.gz      | 0.6          | 1.5          | 3.5          |
| 4   | /path/to/file4.nii.gz      | 0.9          | 1.1          | 3.2          |
| ... | ...                        | ...          | ...          | ...          |
+-----+----------------------------+--------------+--------------+--------------+
```

Prep Output Direction

In [1]:
# Specify where you want to save your results to
out_dir = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/acoe_reliability/results/ICC'

Import Data

In [2]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/acoe_reliability/scores_v1_stacked.csv'
sheet = None

In [3]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
display(data_df)

,subject,scoring_type,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Total
0,Dennis Borschewski,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Dennis Borschewski,M,0,0,0,7,5,4,3,2,...,2,12,4,0,8,3,4,3,4,63
2,Andre Camara,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Andre Camara,M,0,5,2,14,5,4,3,2,...,2,12,4,1,8,4,4,2,4,78
4,Michelle Drad,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Michelle Drad,M,2,0,2,2,2,3,2,0,...,1,10,3,0,4,1,4,0,0,38
6,Nicole Campbell,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Nicole Campbell,M,0,5,3,12,7,4,2,2,...,1,12,4,1,8,4,4,7,5,83
8,Ross Carleton,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Ross Carleton,M,3,0,0,8,0,4,3,2,...,0,1,4,1,5,4,4,1,4,44


# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [4]:
data_df.columns

Index(['subject', 'scoring_type', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9',
       'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19',
       'Q20', 'Total'],
      dtype='object')

In [5]:
drop_list = ['scoring_type', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9',
       'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19',
       'Q20', 'Total']

In [6]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
display(data_df)

,subject,scoring_type,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Total
0,Dennis Borschewski,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Dennis Borschewski,M,0,0,0,7,5,4,3,2,...,2,12,4,0,8,3,4,3,4,63
2,Andre Camara,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Andre Camara,M,0,5,2,14,5,4,3,2,...,2,12,4,1,8,4,4,2,4,78
4,Michelle Drad,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Michelle Drad,M,2,0,2,2,2,3,2,0,...,1,10,3,0,4,1,4,0,0,38
6,Nicole Campbell,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Nicole Campbell,M,0,5,3,12,7,4,2,2,...,1,12,4,1,8,4,4,7,5,83
8,Ross Carleton,A,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Ross Carleton,M,3,0,0,8,0,4,3,2,...,0,1,4,1,5,4,4,1,4,44


**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [7]:
data_df.columns

Index(['subject', 'scoring_type', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9',
       'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19',
       'Q20', 'Total'],
      dtype='object')

Set the parameters for dropping rows

In [8]:
# column = 'City'  # The column you'd like to evaluate
# condition = 'equal'  # The condition to check ('equal', 'above', 'below', 'not')
# value = 'BWH' # The value to drop if found

In [9]:
# data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
# display(data_df)

**Standardize Data**
- Enter Columns you Don't want to standardize into a list

In [10]:
# Remove anything you don't want to standardize
# cols_not_to_standardize = None # ['Z_Scored_Percent_Cognitive_Improvement_By_Origin_Group', 'Z_Scored_Subiculum_T_By_Origin_Group_'] #['Age']

In [11]:
# data_df = cal_palm.standardize_columns(cols_not_to_standardize)
# data_df

In [12]:
# for col in data_df.columns:
#     if 'CSF' and 'eh' not in col:
#         data_df[col] = data_df[col] * -1

# 02 - Derive ICCs

In [13]:
columns_to_compare = ['Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9',
       'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19',
       'Q20', 'Total']
rater_column = 'scoring_type'
y_label = 'Question'

In [14]:
from calvin_utils.statistical_utils.icc_analysis import ICCAnalysis
# Initialize the class
analysis = ICCAnalysis()
# Orchestrate multiple iterations of t-tests and plot boxplots
analysis.orchestrate_icc_analysis(data_df, rater_column, columns_to_compare, outdir=out_dir, icc="ICC(1,1)")

TypeError: ICCAnalysis.orchestrate_icc_analysis() missing 1 required positional argument: 'columns_to_compare'